## 0 Omnilingual Baseline

This notebook gets the baseline Word Error Rate (WER) and Character Error Rate (CER) scores for Meta's Omnilingual model.

In [5]:
import torch
from tqdm import tqdm
from omnilingual_asr.data import load_all_data
from omnilingual_asr.evaluate import add_metrics_columns, idiom_summary, print_evaluation_summary, show_examples
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs
from omnilingual_asr.utils import get_best_gpu, normalize_romansh_text

FileNotFoundError: [Errno 2] No such file or directory: '/local/scratch/matuor/Romansh-ASR/omnilingual/.venv/data/clean-data'

First we set the configuration, the `MODEL_CARD` variable defines the model that we want to evaluate.

In [2]:
MODEL_CARD = "omniASR_CTC_1B_v2"
LANGUAGE_CODE = "roh_Latn_surs1244"
BATCH_SIZE = 8
best_gpu = get_best_gpu()
DEVICE = f"cuda:{best_gpu}" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}, Lang supported: {LANGUAGE_CODE in supported_langs}")

Selected GPU 5 with 24121 MiB free memory
Device: cuda:5, Lang supported: True


Then we load the test data from `data/clean-data`.

In [3]:
df_test = load_all_data("test")
print(f"Loaded {len(df_test)} samples")

Loaded 631 samples


Then we can start transcribing the test set.

In [4]:
pipeline = ASRInferencePipeline(model_card=MODEL_CARD, device=DEVICE)
audio_paths = df_test["audio_path"].tolist()
transcriptions = []

for i in tqdm(range(0, len(audio_paths), BATCH_SIZE)):
    batch = audio_paths[i:i+BATCH_SIZE]
    try:
        results = pipeline.transcribe(batch, lang=[LANGUAGE_CODE]*len(batch), batch_size=len(batch))
        transcriptions.extend(results)
    except Exception as e:
        print(f"Batch error at {i}: {e}")
        transcriptions.extend([""] * len(batch))

df_test["omnilingual_transcription"] = transcriptions

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

 56%|█████▌    | 44/79 [00:32<00:18,  1.91it/s]

Batch error at 344: The map function has failed while processing the path 'data' of the input data. See nested exception for details.


100%|██████████| 79/79 [01:01<00:00,  1.29it/s]


With the transcriptions we can compute the Word Error Rate (WER) and Character Error Rate (CER).

In [5]:
df_test['omnilingual_transcription'] = df_test['omnilingual_transcription'].apply(normalize_romansh_text)
df_test = add_metrics_columns(df_test, "sentence", "omnilingual_transcription")
summary = idiom_summary(df_test)
print_evaluation_summary(summary)


OVERALL RESULTS
Total test samples: 631
Word Error Rate (WER): 64.80%
Character Error Rate (CER): 21.42%

PER IDIOM RESULTS

CC-2021-05-28
  Samples: 81
  WER: 42.61%
  CER: 11.28%

RMPUTER-CC-2021-06-11
  Samples: 114
  WER: 63.71%
  CER: 18.46%

RMSURSILV-CC-2021-05-28
  Samples: 94
  WER: 66.85%
  CER: 20.64%

RMSURSIV-CC-2021-12-23
  Samples: 151
  WER: 70.21%
  CER: 25.42%

RMSUTSILV-CC-2022-05-18
  Samples: 94
  WER: 76.73%
  CER: 29.82%

RMVALLADER-CC-2021-05-28
  Samples: 97
  WER: 63.08%
  CER: 20.06%

SUMMARY TABLE
                   idiom  samples  wer_mean  wer_std  cer_mean  cer_std
           cc-2021-05-28       81     42.61    13.47     11.28     5.50
   rmputer-cc-2021-06-11      114     63.71    26.75     18.46    10.65
 rmsursilv-cc-2021-05-28       94     66.85    13.01     20.64     8.47
  rmsursiv-cc-2021-12-23      151     70.21    21.67     25.42    15.92
 rmsutsilv-cc-2022-05-18       94     76.73    12.89     29.82    15.54
rmvallader-cc-2021-05-28       97   

Finally we can show some example transcriptions.

In [6]:
show_examples(df_test, "omnilingual_transcription")


EXAMPLE TRANSCRIPTIONS

Idiom: rmputer-cc-2021-06-11
Reference:    dasper ils murs vegnan eir churos ils chastagners actuelmaing dad una gruppa da recruts civils chi appredschan la lavur our illa natura
Hypothesis:   daspera ls murs vegnen er ciuros als ciastagners actuelmeinc da d una grupa de recruts zivils ci apregian la lavur ori la natura
WER: 77.3% | CER: 16.3%
----------------------------------------

Idiom: cc-2021-05-28
Reference:    e deblas vischnancas chen mo pli posts executivs dal chantun il referendum duai esser in cler signal cunter adina dapli prescripziuns da surengiu e pe
Hypothesis:   e deblas vishnanchas chen mopli posts executivs dal chantun il referendum duei esser in cler signal cunter adina da pli per scripziuns dasurengiu e pe
WER: 26.3% | CER: 4.6%
----------------------------------------

Idiom: rmsursilv-cc-2021-05-28
Reference:    memia tard vegnevan plitost quels che eran datier dalla scola nus eran adina ad uras nun chei deva in incaps chei era giu lavi